In [5]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k_adjusted.csv to placement_predict_50k_adjusted.csv


In [6]:
import pandas as pd

DATA_PATH = "placement_predict_50k_adjusted.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully! 🎉")

print("\nShape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Dataset loaded successfully! 🎉

Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']

First 5 rows:
   Gender       City CollegeTier      Stream Specialisation Hostel  \
0  Female      Delhi       Tier3          IT    DataScience    Yes   
1    Male    Chennai       Tier2         ECE             AI    Yes   
2  Female  Hyderabad       Tier3         ECE     Networking     No   
3  Female     Jaipur       Tier3         ECE       Embedded     No   
4    Male  Ahmedabad       Tier3  Mechanical    DataScience     No   

  HistoryOfBacklogs  CGPA  AttendancePercent  Internships  ...  Workshops  \
0               Yes  6.63               68.3            2  ...        0.0   
1                No 

In [7]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Column Distribution:")
print(df["PlacementStatus"].value_counts())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Gender              50000 non-null  object 
 1   City                50000 non-null  object 
 2   CollegeTier         50000 non-null  object 
 3   Stream              50000 non-null  object 
 4   Specialisation      50000 non-null  object 
 5   Hostel              50000 non-null  object 
 6   HistoryOfBacklogs   50000 non-null  object 
 7   CGPA                50000 non-null  float64
 8   AttendancePercent   50000 non-null  float64
 9   Internships         50000 non-null  int64  
 10  Projects            50000 non-null  int64  
 11  Workshops           45512 non-null  float64
 12  Certifications      50000 non-null  int64  
 13  Publications        50000 non-null  int64  
 14  AptitudeTestScore   45971 non-null  float64
 15  SoftSkillsRating    46474 non-nu

In [8]:
# -*- coding: utf-8 -*-
"""Adaboosting & XGB.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1vzBljo-kFwAWpSgeg1rF6tcHxfGmbDDR
"""

"""
boosting_benchmark.py
----------------------
Compares AdaBoost (sample-weight boosting) vs XGBoost (gradient boosting with
early stopping) on the PlacementPredict dataset.

Usage:
    python boosting_benchmark.py
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "placement_predict_50k_adjusted.csv"


# --------------------------------------------------------------------------
# 1. Load + preprocess (plain top-level script code, no function wrapper)
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Drop the injected-anomaly flag; it's a data-quality marker, not a real feature
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Label-encode categoricals (fine for tree-based boosters; no dummy blow-up)
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

# Impute missing numerics (median) — AdaBoost's DecisionTree base estimator
# can't handle NaN natively, unlike XGBoost, so both models get the same clean input
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Scale numerics — helps AdaBoost's (default) shallow-tree base estimator converge cleanly
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])


# --------------------------------------------------------------------------
# 2. Train / validation / test split (also plain top-level code)
# --------------------------------------------------------------------------
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# First carve off test, then split remainder into train/val
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio,
    stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


# --------------------------------------------------------------------------
# 3. Benchmark function (kept as a function, since it's meant to be reusable)
# --------------------------------------------------------------------------
def boosting_benchmark(X_train, y_train, X_val, y_val):
    """
    Fits AdaBoost (adaptive sample-weight boosting) and XGBoost
    (gradient boosting with early stopping on the validation set),
    evaluates both on the validation set, and returns a DataFrame
    of results sorted by val_accuracy descending.
    """
    results = []

    # ---- Model 1: AdaBoost ------------------------------------------------
    # AdaBoost re-weights training samples each round: misclassified samples
    # get higher weight so the next weak learner focuses on them.
    ada_base = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
    ada = AdaBoostClassifier(
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE,
    )

    t0 = time.time()
    ada.fit(X_train, y_train)
    ada_fit_time = time.time() - t0

    ada_val_pred = ada.predict(X_val)
    ada_val_proba = ada.predict_proba(X_val)[:, 1]

    results.append({
        "model": "AdaBoost",
        "val_accuracy": accuracy_score(y_val, ada_val_pred),
        "val_f1": f1_score(y_val, ada_val_pred),
        "val_roc_auc": roc_auc_score(y_val, ada_val_proba),
        "best_n_estimators": ada.n_estimators,   # AdaBoost has no early stopping
        "fit_time_sec": round(ada_fit_time, 2),
    })

    # ---- Model 2: XGBoost with early stopping -----------------------------
    # early_stopping_rounds lives on the constructor in xgboost>=2.0; fit()
    # is given eval_set and stops once val logloss hasn't improved in N rounds.
    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    t0 = time.time()
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_fit_time = time.time() - t0

    xgb_val_pred = xgb.predict(X_val)
    xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

    results.append({
        "model": "XGBoost",
        "val_accuracy": accuracy_score(y_val, xgb_val_pred),
        "val_f1": f1_score(y_val, xgb_val_pred),
        "val_roc_auc": roc_auc_score(y_val, xgb_val_proba),
        "best_n_estimators": xgb.best_iteration + 1,  # rounds actually used before stopping
        "fit_time_sec": round(xgb_fit_time, 2),
    })

    results_df = pd.DataFrame(results).sort_values(
        "val_accuracy", ascending=False
    ).reset_index(drop=True)

    return results_df


# --------------------------------------------------------------------------
# 4. Run
# --------------------------------------------------------------------------
leaderboard = boosting_benchmark(X_train, y_train, X_val, y_val)
print("\nValidation leaderboard (sorted by val_accuracy):")
print(leaderboard.to_string(index=False))

leaderboard.to_csv("boosting_benchmark_results.csv", index=False)
print("\nSaved results to boosting_benchmark_results.csv")

Train: (34999, 19) | Val: (7501, 19) | Test: (7500, 19)

Validation leaderboard (sorted by val_accuracy):
   model  val_accuracy   val_f1  val_roc_auc  best_n_estimators  fit_time_sec
AdaBoost      0.796294 0.780963     0.880162                200         11.20
 XGBoost      0.795894 0.782744     0.882638                135          0.88

Saved results to boosting_benchmark_results.csv
